# ⚡ Amazon ML Challenge 2026 — 24-Hour CPU Baseline & Blocking Runner

This notebook is specifically configured for **100% CPU execution on Google Colab** (Standard or High-RAM CPU runtime).

### 💡 Why CPU Runtime is the Best Choice for Blocking & Baseline:
- **Zero GPU Waste**: Stage 1 blocking (inverted token/BM25 indexing), Stage 2c lexical feature extraction, and Stage 3 XGBoost histogram trees run purely on CPU and RAM.
- **24-Hour Stability & No Timeouts**: GPU sessions on Colab have strict quotas and disconnect after 4–6 hours. Standard CPU runtimes run smoothly for up to **12–24 hours** without interruption.
- **Full RAM Utilization**: Uses all 13–16 GB of Colab system RAM with Copy-on-Write (`fork`) process sharing, guaranteeing maximum throughput without memory duplication.
- **Bulletproof Auto-Download**: Automatically packages your entire `output/` directory as a zip, saves a copy directly to Google Drive, and downloads `submission.csv` to your laptop.

## ⚙️ Step 1: System Hardware & RAM Check
Ensure your Colab runtime is set to **CPU** (`Runtime` -> `Change runtime type` -> `None / CPU`).

In [ ]:
import psutil, os, multiprocessing

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
ram_avail_gb = psutil.virtual_memory().available / (1024 ** 3)
cpus = os.cpu_count()

print(f"==================================================")
print(f"  COLAB CPU RUNTIME SPECIFICATIONS")
print(f"==================================================")
print(f"  -> Total CPU Cores:      {cpus}")
print(f"  -> Total System RAM:     {ram_gb:.2f} GB")
print(f"  -> Available Free RAM:   {ram_avail_gb:.2f} GB")
print(f"==================================================")

## 🌐 Step 2: Keep-Alive Prevention (Prevents Browser Sleep)
Run this cell or open your browser console (`Ctrl+Shift+I` or `F12` -> `Console`) and paste the code below to ensure Colab stays connected even if your tab is minimized.

In [ ]:
from IPython.display import display, Javascript

# Auto-click connect button every 60 seconds to prevent idle disconnect
def keep_alive():
    js = '''
    function ClickConnect(){
        console.log("[Keep-Alive] Keeping Colab session active..."); 
        document.querySelector("colab-connect-button").shadowRoot.getElementById('connect').click()
    }
    setInterval(ClickConnect, 60000)
    '''
    display(Javascript(js))

try:
    keep_alive()
    print("Keep-alive script active!")
except Exception as e:
    print(f"Could not inject JS keep-alive: {e}")

## 📥 Step 4: Clone Repository & Pull Git LFS Artifacts
Pulls the codebase, the 2.3 GB pre-computed candidate index, the 343k+ blocking progress, and all dataset files.

In [ ]:
!apt-get install -y git-lfs > /dev/null
!git lfs install

import os
REPO_URL = "https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git"

if not os.path.exists("AmazonMLChallenge2026"):
    print("Cloning repository...")
    !git clone $REPO_URL
    %cd AmazonMLChallenge2026
else:
    %cd AmazonMLChallenge2026
    !git pull origin main

print("\nPulling all large Git LFS artifacts (2.3GB index, datasets, checkpoints)...")
!git lfs pull
print("Git LFS pull complete!")

## 📦 Step 5: Install Dependencies & Verify Environment

In [ ]:
# Install lightweight requirements
!pip install -q -r code/business_entity_resolution/requirements.txt
!chmod +x scripts/*.sh

# Run environment smoke test
!bash scripts/00_verify_environment.sh --python-path python3

## 🚀 Step 6: Start Full CPU Baseline & Blocking Execution

This executes the end-to-end pipeline in **CPU Baseline Mode**:
1. **Stage 1 Blocking**: Loads `candidate_index.pkl` (2.3GB) into RAM, skips the **343k+ already-completed S1 records**, and distributes the remainder across all CPU cores via Linux `fork` Copy-on-Write.
2. **Stage 2c Feature Extraction**: Extracts lexical and deterministic string similarities in parallel.
3. **Stage 3 Scoring**: Fits the calibrated grouped-OOF XGBoost model with monotonic constraints.
4. **Stage 4 & 5 Decision**: Runs injective 1-to-N bipartite clustering and writes `submission.csv`.

In [ ]:
# Launch execution with unbuffered stdout
!python3 -u -c "print('Starting pipeline...')"
!bash scripts/run_all_phases.sh --run-mode Full --skip-gpu --baseline-only --python-path python3

## 📊 Step 7: Validate Submission

In [ ]:
!bash scripts/05_validate_submission.sh --python-path python3

## 🎁 Step 8: Multi-Layer Packaging & Automatic Download

This cell:
1. Directly downloads `submission.csv` to your computer.
2. Creates a clean zip archive `output_baseline_artifacts.zip` containing all stage outputs.
4. Initiates a browser download of the full zip archive to your laptop.

In [ ]:
import os, hashlib
from google.colab import files

# 1. Download submission.csv
sub_file = "output/phase4_predictions/submission.csv"
if os.path.exists(sub_file):
    print(f"[1/3] Downloading {sub_file} to your computer...")
    try:
        files.download(sub_file)
    except Exception as e:
        print(f"Direct browser download skipped ({e}). File remains safe on disk.")
else:
    print(f"Warning: {sub_file} not found.")

# 2. Create output_baseline_artifacts.zip
zip_name = "output_baseline_artifacts.zip"
print(f"\n[2/3] Packaging output/ into {zip_name}...")
!zip -r -q {zip_name} output/ -x "output/**/_tmp_worker_chunks/*" "output/**/*.tmp"

if os.path.exists(zip_name):
    zip_size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"  -> Successfully created {zip_name} ({zip_size_mb:.2f} MB)")
    
    # 4. Prompt browser download of the full zip archive
    print("\nInitiating browser download of the full output archive...")
    try:
        files.download(zip_name)
    except Exception as e:
        print(f"Browser download prompt: {e}")
else:
    print("Error creating zip archive.")